Preparación de datos y primer modelo

En este notebook se prepararán los datos según lo diagnosticado en la etapa de análisis (limpieza de valores vacíos, transformación a números y separación de datos) para entrenar un primer modelo de prueba.

In [74]:
# Para importar las herramientas
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Para cargar el archivo original
df = pd.read_csv("../data/raw/customer_churn_historical.csv")

# Para ver la cantidad de filas y columnas
df.shape

(7043, 21)

In [76]:
# Para quitar columnas que no me sirven
# Para separar los datos de los clientes de la respuesta que queremos predecir (1 se fue, 0 se quedó)
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"Yes": 1, "No": 0})

In [77]:
# Para separar en 80% para entrenar y 20% para probar
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [78]:
# Para ver cuántos clientes quedaron para cada parte
print(f"Para entrenar: {X_train.shape[0]}")
print(f"Para probar: {X_test.shape[0]}")

Para entrenar: 5634
Para probar: 1409


* Se descartó "customerID" porque es un código que no aporta información para predecir el comportamiento del cliente.
* Se descartó "Churn" en X para que el modelo no tenga la respuesta de antemano, y aprenda a deducirla a partir del resto de los datos.
* Se separó "Churn" como la columna objetivo a predecir, transformando sus valores a 1 (se fue) y 0 (se quedó).
* Se dividieron los datos en 5634 filas para entrenamiento (80%) y 1409 para prueba (20%), manteniendo en ambas partes la misma proporción de clientes que cancelaron el servicio.

In [79]:
# Para separar las columnas numéricas de las categóricas

columnas_numericas = X_train.select_dtypes(include=["int64", "float64"]).columns

columnas_categoricas = X_train.select_dtypes(include=["object", "str"]).columns

print("Columnas numéricas:")
print(columnas_numericas.tolist())

print("\nColumnas categóricas:")
print(columnas_categoricas.tolist())

Columnas numéricas:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Columnas categóricas:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [80]:
# Para importar las herramientas de preprocesamiento

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [81]:
# Para preparar las columnas numéricas

pipeline_numerico = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler())
])

In [82]:
# Para importar la herramienta que transforma las categorías en números

from sklearn.preprocessing import OneHotEncoder

In [83]:
# Para preparar las columnas categóricas

pipeline_categorico = Pipeline([
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("codificacion", OneHotEncoder(handle_unknown="ignore"))
])

In [84]:
# Para importar la herramienta que aplica distintos procesos según las columnas

from sklearn.compose import ColumnTransformer

In [85]:
# Para unir la perparación de las columnas numéricas y categóricas

preprocesamiento = ColumnTransformer([
    ("numericas", pipeline_numerico, columnas_numericas),
    ("categoricas", pipeline_categorico, columnas_categoricas)
])

In [86]:
# Para importar el modelo que se va a usar como referencia

from sklearn.dummy import DummyClassifier

In [87]:
# Para crear el modelo de referencia junto con el preprocesamiento

modelo_baseline = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", DummyClassifier(strategy="most_frequent"))
])

In [88]:
# Para entrenar el modelo de referencia

modelo_baseline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specify

In [89]:
# Para generar las predicciones sobre el conjunto de prueba

y_pred_baseline = modelo_baseline.predict(X_test)

In [90]:
# Para importar las herramientas de evaluación

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [91]:
# Para mostrar la matriz de confusión y el reporte de clasificación

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_baseline))

Matriz de confusión:
[[1037    0]
 [ 372    0]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.74      1.00      0.85      1037
           1       0.00      0.00      0.00       372

    accuracy                           0.74      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.74      0.62      1409



c:\Users\santi\Documents\Laboratorio_Mineria\Proyecto\proyecto_churn\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\santi\Documents\Laboratorio_Mineria\Proyecto\proyecto_churn\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\santi\Documents\Laboratorio_Mineria\Proyecto\proyecto_churn\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d

In [92]:
# Para calcular ROC-AUC usando las probabilidades del modelo

y_prob_baseline = modelo_baseline.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_baseline))

ROC-AUC: 0.5


Resultado del modelo baseline:

El modelo baseline predijo que todos los clientes permanecerían en la empresa, ya que ellos son los más frecuentes.
Aunque obtuvo alrededor de un 74% de accuracy, no pudo identificar a ninguno de los clientes que abandonaron el servicio, y por eso, el recall para la clase Churn fue 0.
Este resultado muestra que la accuracy por sí misma no es suficiente para evaluar el problema, ya que un modelo puede obtener un valor relativamente alto sin detectar ningún caso de abandono.

In [93]:
# Para importar el modelo de Regresión Logística

from sklearn.linear_model import LogisticRegression

In [94]:
# Para crear el modelo de Regresión Logística junto con el preprocesamiento

modelo_logistico = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", LogisticRegression(max_iter=1000))
])

In [95]:
# Para entrenar el modelo de Regresión Logística

modelo_logistico.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specify

In [96]:
# Para generar las predicciones sobre el conjunto de prueba

y_pred_logistico = modelo_logistico.predict(X_test)

In [97]:
# Para mostrar la matriz de confusión y el reporte de clasificación

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_logistico))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_logistico, zero_division=0))

Matriz de confusión:
[[952  85]
 [205 167]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1037
           1       0.66      0.45      0.54       372

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409



In [98]:
# Para calcular ROC-AUC usando las probabilidades del modelo

y_prob_logistico = modelo_logistico.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_logistico))

ROC-AUC: 0.8119601621716905


Resultado de la Regresión Logística:

La Regresión Logística obtuvo una accuracy de aproximadamente 79% y un ROC-AUC de 0.81.
De los 372 clientes que realmente abandonaron el servicio, el modelo identificó correctamente a 167 y no detectó a 205. Esto produjo un recall de 0.45 para la clase Churn.
Comparado con el modelo baseline, la Regresión Logística logra detectar parte de los clientes que abandonan y presenta una mayor capacidad para diferenciar entre ambas clases, aunque todavía deja sin detectar una cantidad importante de casos de abandono.

In [99]:
# Para importar el modelo Random Forest

from sklearn.ensemble import RandomForestClassifier

In [100]:
# Para crear el modelo Random Forest junto con el preprocesamiento

modelo_random_forest = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("modelo", RandomForestClassifier(random_state=42))
])

In [101]:
# Para entrenar el modelo Random Forest

modelo_random_forest.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specify

In [102]:
# Para generar las predicciones sobre el conjunto de prueba

y_pred_random_forest = modelo_random_forest.predict(X_test)

In [103]:
# Para mostrar la matriz de confusión y el reporte de clasificación

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_random_forest))

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_random_forest, zero_division=0))

Matriz de confusión:
[[954  83]
 [223 149]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.81      0.92      0.86      1037
           1       0.64      0.40      0.49       372

    accuracy                           0.78      1409
   macro avg       0.73      0.66      0.68      1409
weighted avg       0.77      0.78      0.76      1409



In [104]:
# Para calcular ROC-AUC usando las probabilidades del modelo

y_prob_random_forest = modelo_random_forest.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_random_forest))

ROC-AUC: 0.7901748219118425


Resultado de Random Forest:

Random Forest obtuvo una accuracy de aproximadamente 78% y un ROC-AUC de 0.79.
De los 372 clientes que realmente abandonaron el servicio, el modelo identificó correctamente a 149 y no detectó a 223, obteniendo un recall de 0.40 para la clase Churn.
En esta primera comparación, Random Forest obtuvo resultados inferiores a la Regresión Logística, especialmente en recall, F1-score y ROC-AUC.

In [105]:
# Para importar las métricas necesarias para comparar los modelos

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [106]:
# Para comparar los resultados de los tres modelos

resultados = pd.DataFrame({
    "Modelo": ["Baseline", "Regresión Logística", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_baseline),
        accuracy_score(y_test, y_pred_logistico),
        accuracy_score(y_test, y_pred_random_forest)
    ],
    "Precision": [
        precision_score(y_test, y_pred_baseline, zero_division=0),
        precision_score(y_test, y_pred_logistico, zero_division=0),
        precision_score(y_test, y_pred_random_forest, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_baseline, zero_division=0),
        recall_score(y_test, y_pred_logistico, zero_division=0),
        recall_score(y_test, y_pred_random_forest, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, y_pred_baseline, zero_division=0),
        f1_score(y_test, y_pred_logistico, zero_division=0),
        f1_score(y_test, y_pred_random_forest, zero_division=0)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_baseline),
        roc_auc_score(y_test, y_prob_logistico),
        roc_auc_score(y_test, y_prob_random_forest)
    ]
})

resultados.round(3)

,Modelo,Accuracy,Precision,Recall,F1,ROC-AUC
0,Baseline,0.736,0.000,0.000,0.000,0.500
1,Regresión Logística,0.794,0.663,0.449,0.535,0.812
2,Random Forest,0.783,0.642,0.401,0.493,0.790


Comparación de modelos:

Se compararon 3 modelos: un baseline, Regresión Logística y Random Forest.
El baseline obtuvo una accuracy de 0.736, pero no logró detectar ningún caso de abandono, por lo que su recall para Churn fue 0.
La Regresión Logística obtuvo los mejores resultados de esta primera comparación, con una accuracy de 0.794, un recall de 0.449, un F1-score de 0.535 y un ROC-AUC de 0.812.
Random Forest obtuvo una accuracy de 0.783, un recall de 0.401, un F1-score de 0.493 y un ROC-AUC de 0.790.
Por el momento, la Regresión Logística presenta los mejores resultados entre los modelos evaluados.
Sin embargo, todavía deja sin detectar una cantidad importante de clientes que abandonan el servicio.